In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path

from analysis.utils.utils import get_weights_path, get_figures_path

# ── Config ────────────────────────────────────────────────────────────────────
HISTORY_FILE = get_weights_path() / "eval_history.jsonl"

# Leave empty to compare ALL runs, or list weights_dir strings to filter:
# e.g. COMPARE = ["gravnet_regression_faser_all_events_mean_reparam_std",
#                 "gravnet_regression_faser_all_events_mean_reparam_std_huber1.0"]
COMPARE = []

TARGETS      = ["E_nu", "E_lepton", "E_roe"]
TARGET_LATEX = [r"$E_\nu$", r"$E_\mathrm{lep}$", r"$E_\mathrm{roe}$"]
ENERGY_BINS_TEV = [
    (0.01, 0.05), (0.05, 0.1), (0.1, 0.2),
    (0.2,  0.3),  (0.3,  0.5), (0.5,  0.7),
    (0.7,  1.0),  (1.0,  1.5), (1.5,  3.0),
]

sns.set_style("ticks")
sns.set_context("paper", font_scale=1.2)
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['DejaVu Serif', 'Times New Roman', 'Times'],
    'mathtext.fontset': 'dejavuserif',
    'axes.linewidth': 0.8,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'figure.dpi': 200,
})

In [ ]:
# ── Logic to handle auto-selection ──────────────────────────────────────────
if not COMPARE:
    try:
        all_entries = []
        with open(HISTORY_FILE, "r") as f:
            for line in f:
                if line.strip():
                    all_entries.append(json.loads(line))
        
        # Sort by timestamp (ISO format strings sort correctly)
        all_entries.sort(key=lambda x: x["timestamp"])
        
        # Get unique weights_dir values while preserving recent order
        unique_dirs = []
        for entry in reversed(all_entries):
            wd = entry["weights_dir"]
            if wd not in unique_dirs:
                unique_dirs.append(wd)
            if len(unique_dirs) == 2:
                break
        
        COMPARE = unique_dirs
        print(f"Auto-selected recent runs: {COMPARE}")
        
    except FileNotFoundError:
        print(f"Warning: {HISTORY_FILE} not found.")
    except Exception as e:
        print(f"Error parsing history: {e}")


In [ ]:
def short_name(weights_dir):
    """Strip common prefix for display."""
    prefix = "gravnet_regression_faser_"
    return weights_dir.replace(prefix, "") if weights_dir.startswith(prefix) else weights_dir

# ── Load history ──────────────────────────────────────────────────────────────
all_runs = [json.loads(l) for l in open(HISTORY_FILE) if l.strip()]

# Deduplicate by cache_key (keep last occurrence)
seen = {}
for r in all_runs:
    seen[r["cache_key"]] = r
all_runs = list(seen.values())

if COMPARE:
    # Keep only the most recent entry per weights_dir, in the order of COMPARE
    by_wd = {}
    for r in all_runs:
        wd = r["weights_dir"]
        if wd in COMPARE:
            if wd not in by_wd or r["timestamp"] > by_wd[wd]["timestamp"]:
                by_wd[wd] = r
    runs = [by_wd[wd] for wd in COMPARE if wd in by_wd]
else:
    runs = all_runs

print(f"Loaded {len(all_runs)} unique entries from history, comparing {len(runs)}.")
print()
for i, r in enumerate(runs):
    c = r["checkpoint"]
    loss = c.get("loss_fn") or "mse"
    delta = f"δ={c['huber_delta']}" if loss == "huber" and c.get("huber_delta") else ""
    print(f"[{i}] {r['weights_dir']}")
    print(f"     loss={loss}{delta}  epoch={c['epoch']}  val_loss={c['val_loss']:.4f}  val_rmse={c['val_rmse']:.4f}  ts={r['timestamp']}")

import os
_names = [short_name(r['weights_dir']) for r in runs]
_prefix = os.path.commonprefix(_names)
_col_names = [n[len(_prefix):].strip('_') or 'base' for n in _names]

_comparison_key = "_vs_".join(_col_names)
_figures_path = get_figures_path() / "comparisons" / _comparison_key
_figures_path.mkdir(parents=True, exist_ok=True)
print(f"\nFigures → {_figures_path}")

In [ ]:
GREEN, RED, RESET = "\033[32m", "\033[31m", "\033[0m"

higher_is_better = {"r2": True, "median_rel_err": False, "std_res": False, "mean_bias": False}

def _delta_row(label_a, label_b, v0, v1, metric):
    vals = "".join(f"{v:>20.4f}" for v in [v0, v1])
    if len(runs) >= 2:
        diff = v1 - v0
        if metric == "mean_bias":
            good = abs(v1) < abs(v0)
        else:
            good = diff > 0 if higher_is_better[metric] else diff < 0
        colour = GREEN if good else RED
        diff_str = f"{colour}{diff:>+20.4f}{RESET}"
    else:
        diff_str = ""
    print(f"{label_a:<12} {label_b:<14}{vals}{diff_str}")

print(f"{'Target':<12} {'Metric':<14}" + "".join(f"{c:>20}" for c in _col_names) + f"{'Δ (1-0)':>20}")
print("-" * (26 + 20 * (len(runs) + 1)))
for t in TARGETS:
    for metric, label in [("r2", "R²"), ("median_rel_err", "Med|RelErr|"), ("std_res", "Std(res)"), ("mean_bias", "Bias")]:
        _delta_row(t, label, runs[0]['overall'][t][metric], runs[1]['overall'][t][metric] if len(runs) >= 2 else 0, metric)
    print()

# y = E_roe/E_nu  (Bjorken inelasticity = 1 - lepton fraction)
print(f"{'Lep frac y':<12} {'Metric':<14}" + "".join(f"{c:>20}" for c in _col_names) + f"{'Δ (1-0)':>20}")
print("-" * (26 + 20 * (len(runs) + 1)))
for metric, label in [("r2", "R²"), ("std_res", "Std(res)"), ("mean_bias", "Bias")]:
    _delta_row("Inelasticity y", label, runs[0]['inelasticity'][metric], runs[1]['inelasticity'][metric] if len(runs) >= 2 else 0, metric)

print()
print("── Metric legend ───────────────────────────────────────────────────────────")
print("  R²           Coefficient of determination.  1 = perfect.           ↑ better")
print("  Med|RelErr|  Median |pred − true| / true.   0 = perfect.           ↓ better")
print("  Std(res)     Std dev of (pred − true) / true (resolution).         ↓ better")
print("  Bias         Mean    (pred − true) / true   (systematic offset).   → 0")
print("  Δ (1−0)      Change from run 0 to run 1.    Green = improvement.")
print("  Lep frac y   y = E_lep/E_nu (lepton fraction = 1 − y_Bjorken).")
print("────────────────────────────────────────────────────────────────────────────")

In [ ]:
# ── Bar chart: overall metrics per target ─────────────────────────────────────
metrics_to_plot = [
    ("r2",             "R²",           "↑"),
    ("median_rel_err", "Med|RelErr|",   "↓"),
    ("std_res",        "Std(res)",      "↓"),
    ("mean_bias",      "Mean bias",     "→0"),
]

palette = sns.color_palette("BrBG", n_colors=len(runs))
x = np.arange(len(TARGETS))
width = 0.8 / len(runs)

fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(5 * len(metrics_to_plot), 5))

for ax, (metric, label, indicator) in zip(axes, metrics_to_plot):
    for i, r in enumerate(runs):
        vals = [r["overall"][t][metric] for t in TARGETS]
        offset = (i - len(runs) / 2 + 0.5) * width
        bars = ax.bar(x + offset, vals, width * 0.9,
                      label=short_name(r["weights_dir"]),
                      color=palette[i], alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(TARGET_LATEX)
    ax.set_ylabel(label)
    ax.set_title(f"{label} ({indicator} better)")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(True, axis="y", alpha=0.3)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=min(len(runs), 3),
           bbox_to_anchor=(0.5, -0.12), frameon=False, fontsize=9)
plt.tight_layout()
plt.savefig(_figures_path / "bar_metrics.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {_figures_path / 'bar_metrics.png'}")

In [ ]:
import torch
import matplotlib.colors as mcolors

def _preds_to_physical(preds_raw, norm_stats):
    preds = torch.from_numpy(preds_raw)
    if norm_stats is not None:
        log_E_nu = preds[:, 0] * norm_stats["sigma_enu"]   + norm_stats["mu_enu"]
        logit_y  = preds[:, 1] * norm_stats["sigma_logit"] + norm_stats["mu_logit"]
    else:
        log_E_nu = preds[:, 0]
        logit_y  = preds[:, 1]
    E_nu  = 10 ** log_E_nu
    y     = torch.sigmoid(logit_y)
    E_roe = y * E_nu          # y = inelasticity = E_roe/E_nu
    E_lep = (1 - y) * E_nu
    return torch.stack([E_nu, E_lep, E_roe], dim=1).numpy(), y.numpy()

weights_path_base = get_weights_path()
run_data = []
for i, r in enumerate(runs):
    ckpt  = torch.load(weights_path_base / r["weights_dir"] / "best_model.pt", map_location="cpu", weights_only=False)
    cache = np.load(weights_path_base / r["weights_dir"] / f"inference_cache_{r['cache_key']}.npz")
    preds_linear, y_pred = _preds_to_physical(cache["preds_raw"], ckpt.get("norm_stats"))
    targets_linear = cache["targets_linear"]
    run_data.append({
        "col_name":       _col_names[i],
        "preds_linear":   preds_linear,
        "targets_linear": targets_linear,
        "y_pred":         y_pred,
        "y_true":         targets_linear[:, 2] / targets_linear[:, 0].clip(min=1e-6),
    })

# 4 parity plots per run: E_nu, y (lepton fraction), E_lep, E_roe
plot_targets = [
    ("E_nu",         r"$E_\nu$",                                        "TeV", True),
    ("inelasticity", r"$y = E_\mathrm{roe}/E_\nu$  (Bjorken inelasticity)", "", False),
    ("E_lepton",     r"$E_\mathrm{lep}$",                               "TeV", True),
    ("E_roe",        r"$E_\mathrm{roe}$",                               "TeV", True),
]

cmap_parity = mcolors.LinearSegmentedColormap.from_list(
    'trunc', plt.cm.ocean(np.linspace(0.3, 0.9, 100)))

fig, axes = plt.subplots(len(run_data), len(plot_targets),
                         figsize=(6 * len(plot_targets), 5 * len(run_data)))
if len(run_data) == 1:
    axes = axes.reshape(1, -1)

for row_i, rd in enumerate(run_data):
    tl = rd["targets_linear"]
    pl = rd["preds_linear"]

    for col_i, (key, latex, units, log_scale) in enumerate(plot_targets):
        ax = axes[row_i, col_i]

        if key == "inelasticity":
            y_true_p = rd["y_true"]
            y_pred_p = rd["y_pred"]
        elif key == "E_nu":
            y_true_p, y_pred_p = tl[:, 0], pl[:, 0]
        elif key == "E_lepton":
            y_true_p, y_pred_p = tl[:, 1], pl[:, 1]
        else:
            y_true_p, y_pred_p = tl[:, 2], pl[:, 2]

        ss_res    = np.sum((y_true_p - y_pred_p) ** 2)
        ss_tot    = np.sum((y_true_p - y_true_p.mean()) ** 2)
        r2        = 1 - ss_res / ss_tot
        pearson_r = np.corrcoef(y_true_p, y_pred_p)[0, 1]

        ax.set_facecolor("#F5F5F5")

        if log_scale:
            lo = max(y_true_p.min(), 1e-6)
            hi = y_true_p.max()
            hb = ax.hexbin(y_true_p, y_pred_p,
                           xscale='log', yscale='log',
                           gridsize=120, cmap=cmap_parity, bins='log',
                           mincnt=1, alpha=0.8,
                           extent=[np.log10(lo), np.log10(hi),
                                   np.log10(lo), np.log10(hi)])
            ax.set_xlim(lo, hi)
            ax.set_ylim(lo, hi)
            ax.plot([lo, hi], [lo, hi], "k--", linewidth=0.5)
        else:
            pred_limit = float(np.percentile(y_pred_p, 99))
            hi_i = min(max(np.percentile(y_true_p, 99), pred_limit), 3.0)
            pad  = 0.05
            hb = ax.hexbin(y_true_p, y_pred_p,
                           gridsize=100, cmap=cmap_parity, bins='log',
                           mincnt=1, alpha=0.8,
                           extent=[0, hi_i, 0, hi_i])
            ax.set_xlim(-pad, hi_i + pad)
            ax.set_ylim(-pad, hi_i + pad)
            ax.plot([0, hi_i], [0, hi_i], "k--", linewidth=0.5)

        plt.colorbar(hb, ax=ax, label='Counts')
        ax.set_aspect('equal')
        xlabel = rf"True {latex}" + (f" [{units}]" if units else "")
        ylabel = rf"Predicted {latex}" + (f" [{units}]" if units else "")
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_title(f"{rd['col_name']} — {key.replace('_', ' ')}")
        ax.text(0.05, 0.95, f"$R^2$ = {r2:.4f}\n$r$ = {pearson_r:.4f}",
                transform=ax.transAxes, va="top", fontsize=9,
                bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
        ax.spines[['top', 'right']].set_visible(False)
        ax.grid(True, linestyle=':', linewidth=0.8, color='gray', alpha=0.3)

plt.tight_layout()
plt.savefig(_figures_path / "parity_comparison.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {_figures_path / 'parity_comparison.png'}")

In [ ]:
# ── Per-bin resolution (Gaussian σ) ──────────────────────────────────────────
bin_centers = [np.sqrt(emin * emax) for emin, emax in ENERGY_BINS_TEV]
bin_labels  = [f"{emin}-{emax}" for emin, emax in ENERGY_BINS_TEV]

fig, axes = plt.subplots(1, len(TARGETS), figsize=(6 * len(TARGETS), 5))

for ax, target, latex in zip(axes, TARGETS, TARGET_LATEX):
    for i, r in enumerate(runs):
        bins = r["per_bin"][target]
        xs = [np.sqrt(b["emin"] * b["emax"]) for b in bins]
        # prefer gauss_sigma, fall back to std_res
        ys = [b["gauss_sigma"] if b["gauss_sigma"] is not None else b["std_res"] for b in bins]
        ax.plot(xs, ys, "o-", color=palette[i], linewidth=1.2, markersize=5,
                markerfacecolor="white", markeredgewidth=1.2,
                label=short_name(r["weights_dir"]))
    ax.set_xscale("log")
    ax.set_xlabel(rf"True {latex} [TeV]")
    ax.set_ylabel(r"Resolution $\sigma$ (Gaussian fit)")
    ax.set_title(target)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(True, linestyle=":", alpha=0.8)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=min(len(runs), 3),
           bbox_to_anchor=(0.5, -0.12), frameon=False, fontsize=9)
plt.suptitle("Per-bin resolution", fontsize=13)
plt.tight_layout()
plt.savefig(_figures_path / "resolution.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {_figures_path / 'resolution.png'}")

In [ ]:
# ── Per-bin bias (mean residual) ──────────────────────────────────────────────
fig, axes = plt.subplots(1, len(TARGETS), figsize=(6 * len(TARGETS), 5))

for ax, target, latex in zip(axes, TARGETS, TARGET_LATEX):
    for i, r in enumerate(runs):
        bins = r["per_bin"][target]
        xs = [np.sqrt(b["emin"] * b["emax"]) for b in bins]
        ys = [b["mean_res"] for b in bins]
        ax.plot(xs, ys, "o-", color=palette[i], linewidth=1.2, markersize=5,
                markerfacecolor="white", markeredgewidth=1.2,
                label=short_name(r["weights_dir"]))
    ax.axhline(0, color="k", linestyle=":", linewidth=0.8, alpha=0.8)
    ax.set_xscale("log")
    ax.set_xlabel(rf"True {latex} [TeV]")
    ax.set_ylabel("Mean (pred - true) / true")
    ax.set_title(target)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(True, linestyle=":", alpha=0.8)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=min(len(runs), 3),
           bbox_to_anchor=(0.5, -0.12), frameon=False, fontsize=9)
plt.suptitle("Per-bin bias", fontsize=13)
plt.tight_layout()
plt.savefig(_figures_path / "bias.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {_figures_path / 'bias.png'}")

In [ ]:
import pandas as pd

def _build_df(rd):
    tl = rd["targets_linear"]
    pl = rd["preds_linear"]
    y_true = rd["y_true"]
    y_pred = rd["y_pred"]
    rows = []
    for i in range(len(tl)):
        rows.append({
            "E_nu":        tl[i, 0],
            "y_true":      y_true[i],
            "abserr_Enu":  abs((pl[i, 0] - tl[i, 0]) / max(tl[i, 0], 1e-6)),
            "abserr_Elep": abs((pl[i, 1] - tl[i, 1]) / max(tl[i, 1], 1e-6)),
            "abserr_Eroe": abs((pl[i, 2] - tl[i, 2]) / max(tl[i, 2], 1e-6)),
            "abserr_y":    abs((y_pred[i] - y_true[i]) / max(y_true[i], 1e-6)),
        })
    df = pd.DataFrame(rows)
    Q = 0.25
    df["worst_Enu"]      = df["abserr_Enu"] >= df["abserr_Enu"].quantile(1 - Q)
    df["worst_y"]        = df["abserr_y"]   >= df["abserr_y"].quantile(1 - Q)
    df["best_Enu"]       = df["abserr_Enu"] <= df["abserr_Enu"].quantile(Q)
    df["best_y"]         = df["abserr_y"]   <= df["abserr_y"].quantile(Q)
    df["worst_Enu_only"] = df["worst_Enu"] & ~df["worst_y"]
    df["worst_y_only"]   = df["worst_y"]   & ~df["worst_Enu"]
    df["worst_both"]     = df["worst_Enu"] & df["worst_y"]
    df["best_both"]      = df["best_Enu"]  & df["best_y"]
    return df

run_dfs = [_build_df(rd) for rd in run_data]

# ── Error correlation: one scatter per run ─────────────────────────────────────
fig, axes = plt.subplots(1, len(run_data), figsize=(6 * len(run_data), 5),
                         sharey=True)
if len(run_data) == 1:
    axes = [axes]

for ax, df, rd in zip(axes, run_dfs, run_data):
    r = df["abserr_Enu"].corr(df["abserr_y"])
    xlim = np.percentile(df["abserr_Enu"], 99)
    ylim = np.percentile(df["abserr_y"],   99)
    ax.scatter(df["abserr_Enu"], df["abserr_y"],
               s=3, alpha=0.5, color=palette[run_data.index(rd)], linewidths=0)
    ax.axvline(df["abserr_Enu"].quantile(0.75), color="#e05c5c",
               linestyle="--", linewidth=1.0, label="Worst $E_\\nu$ (75th pct)")
    ax.axhline(df["abserr_y"].quantile(0.75),   color="#e8a838",
               linestyle="--", linewidth=1.0, label=r"Worst $y$ (75th pct)")
    ax.set_xlim(0, xlim)
    ax.set_ylim(0, ylim)
    ax.set_xlabel(r"$|$rel err$|$  $E_\nu$")
    ax.set_ylabel(r"$|$rel err$|$  $y = E_\mathrm{lep}/E_\nu$")
    ax.set_title(f"{rd['col_name']}  ($r={r:.3f}$)")
    ax.legend(fontsize=7, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(True, linestyle=":", alpha=0.3)

plt.suptitle(r"Error correlation: $E_\nu$ vs lepton fraction $y = E_\mathrm{lep}/E_\nu$  (= $1 - y_\mathrm{Bjorken}$)", fontsize=12)
plt.tight_layout()
plt.savefig(_figures_path / "error_correlation.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {_figures_path / 'error_correlation.png'}")

In [ ]:
# ── Failure mode breakdown: % events per group, per run ───────────────────────
group_cols   = ["best_both", "worst_Enu_only", "worst_y_only", "worst_both"]
group_labels = ["Best both", "Worst $E_\\nu$ only", "Worst $y$ only", "Worst both"]
group_colors = ["#5b8db8", "#8dbd8d", "#e8a838", "#e05c5c"]

pcts = np.array([[100 * df[g].mean() for g in group_cols] for df in run_dfs])  # [n_runs, 4]

x     = np.arange(len(group_labels))
width = 0.8 / len(runs)

fig, ax = plt.subplots(figsize=(9, 5))
for i, (rd, row) in enumerate(zip(run_data, pcts)):
    offset = (i - len(runs) / 2 + 0.5) * width
    bars = ax.bar(x + offset, row, width * 0.9,
                  label=rd["col_name"], color=palette[i], alpha=0.85)
    for bar, v in zip(bars, row):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f"{v:.1f}%", ha="center", va="bottom", fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels(group_labels, fontsize=9)
ax.set_ylabel("% of val events")
ax.set_title("Failure mode breakdown per run  (25th / 75th percentile thresholds)")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(True, axis="y", linestyle=":", alpha=0.3)
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.savefig(_figures_path / "failure_modes.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {_figures_path / 'failure_modes.png'}")

# Numeric summary
print(f"\n{'Group':<22}" + "".join(f"{rd['col_name']:>16}" for rd in run_data))
for j, lbl in enumerate(group_labels):
    row_str = f"{lbl.replace('$',''):<22}" + "".join(f"{pcts[i, j]:>15.1f}%" for i in range(len(runs)))
    print(row_str)

In [38]:
# ── GravNet vs linear baseline ────────────────────────────────────────────────
print(f"{'Run':<40} {'Target':<12} {'GravNet σ':>10} {'Linear σ':>10} {'Improvement':>12}")
print("-" * 86)
for r in runs:
    name = short_name(r["weights_dir"])
    for t in TARGETS:
        b = r["baseline"][t]
        imp = (b["linear_std_res"] - b["gravnet_std_res"]) / b["linear_std_res"] * 100
        print(f"{name:<40} {t:<12} {b['gravnet_std_res']:>10.3f} {b['linear_std_res']:>10.3f} {imp:>+11.1f}%")
    print()

Run                                      Target        GravNet σ   Linear σ  Improvement
--------------------------------------------------------------------------------------
all_events_mean_reparam_std              E_nu              0.113      0.140       +19.1%
all_events_mean_reparam_std              E_lepton          5.757     12.061       +52.3%
all_events_mean_reparam_std              E_roe             3.957     35.538       +88.9%

all_events_mean_reparam_std_huber1.0     E_nu              0.144      0.140        -2.9%
all_events_mean_reparam_std_huber1.0     E_lepton          5.476     12.061       +54.6%
all_events_mean_reparam_std_huber1.0     E_roe             5.613     35.538       +84.2%

